In [14]:
import pandas as pd

PATH_XGB  = "./data/xgb_5.csv"
PATH_LGBM = "./data/LGBM2.csv"
PATH_CATB = "./data/catb.csv"

def blend_mean(xgb_path=PATH_XGB, lgbm_path=PATH_LGBM, catb_path=PATH_CATB,
               weighted=None, clip0=True):
    def load(path, name):
        df = pd.read_csv(path)
        key2 = "영업장명_메뉴명" if "영업장명_메뉴명" in df.columns else "영업장명_메뉴"
        df = df[["영업일자", key2, "매출수량"]].copy()
        df["매출수량"] = df["매출수량"].astype("float64")   # 평균 계산 위해 float
        df.rename(columns={"매출수량": name, key2: "영업장명_메뉴명"}, inplace=True)
        return df

    xgb  = load(xgb_path,  "xgb")
    lgbm = load(lgbm_path, "lgbm")
    catb = load(catb_path, "catb")

    keys = ["영업일자", "영업장명_메뉴명"]
    m = xgb.merge(lgbm, on=keys).merge(catb, on=keys)

    if clip0:
        m[["xgb","lgbm","catb"]] = m[["xgb","lgbm","catb"]].clip(lower=0)

    if weighted is None:
        m["매출수량"] = m[["xgb","lgbm","catb"]].mean(axis=1)
    else:
        wx, wl, wc = weighted
        m["매출수량"] = wx*m["xgb"] + wl*m["lgbm"] + wc*m["catb"]

    return m[keys + ["매출수량"]]

In [16]:
# 단순 평균
# sub_mean = blend_mean()
# 가중 평균 (xgb, lgbm, catb)
sub_weighted = blend_mean(weighted=(0.75, 0.0, 0.25))

In [18]:
def convert_to_submission_format(pred_df: pd.DataFrame, sample_submission: pd.DataFrame):
    pred_dict = dict(zip(
        zip(pred_df['영업일자'], pred_df['영업장명_메뉴명']),
        pred_df['매출수량']
    ))
    final_df = sample_submission.copy()
    for row_idx in final_df.index:
        date = final_df.loc[row_idx, '영업일자']
        for col in final_df.columns[1:]:
            final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
    return final_df

In [20]:
sample_submission = pd.read_csv('./data/sample_submission.csv')
submission = convert_to_submission_format(sub_weighted, sample_submission)
submission.head()

C:\Users\user\AppData\Local\Temp\ipykernel_832\2836532793.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7.2950625447744715' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
C:\Users\user\AppData\Local\Temp\ipykernel_832\2836532793.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.0604588364985466' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
C:\Users\user\AppData\Local\Temp\ipykernel_832\2836532793.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '5.718909352821274' has dtype incompatible with int64, please explicitly cast to a 

,영업일자,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ_BBQ55(단체),"느티나무 셀프BBQ_대여료 30,000원","느티나무 셀프BBQ_대여료 60,000원","느티나무 셀프BBQ_대여료 90,000원","느티나무 셀프BBQ_본삼겹 (단품,실내)",느티나무 셀프BBQ_스프라이트 (단체),느티나무 셀프BBQ_신라면,느티나무 셀프BBQ_쌈야채세트,...,화담숲주막_스프라이트,화담숲주막_참살이 막걸리,화담숲주막_찹쌀식혜,화담숲주막_콜라,화담숲주막_해물파전,화담숲카페_메밀미숫가루,화담숲카페_아메리카노 HOT,화담숲카페_아메리카노 ICE,화담숲카페_카페라떼 ICE,화담숲카페_현미뻥스크림
0,TEST_00+1일,7.295063,3.060459,5.718909,3.237218,0.792509,1.319488,4.921197,2.801374,1.659368,...,5.382799,9.166550,8.789964,5.191742,42.893928,32.398949,3.429684,24.200739,6.979762,13.766292
1,TEST_00+2일,3.262123,32.392687,2.898582,2.008762,0.762835,1.125382,2.586151,2.019966,1.492080,...,2.624677,3.813991,3.488024,2.700943,15.448299,8.900894,1.681836,7.563652,2.855774,4.924533
2,TEST_00+3일,3.368939,72.709878,2.271607,1.436244,0.731547,1.119314,3.947853,1.713637,1.395938,...,2.026998,3.526485,3.543364,2.241690,13.811640,10.792960,1.761203,8.969966,3.105638,5.063214
3,TEST_00+4일,3.456828,23.610083,2.329456,1.642629,0.712929,1.253859,13.258536,1.877618,1.601622,...,1.854810,3.711126,3.823017,2.114227,15.144177,9.670003,1.532332,8.287880,2.642191,4.777901
4,TEST_00+5일,3.138946,28.355280,2.450261,2.061659,0.897156,1.472262,11.054083,1.968043,2.081280,...,2.461858,3.963486,3.479316,2.405144,12.525529,9.609405,1.829666,10.673553,2.537197,4.937120


In [22]:
submission.to_csv('Gonjiam_submission_ensemble_w13.csv', index=False, encoding='utf-8-sig')